# Лабораторная №3. Распознавание объектов на изображениях Fashion MNIST

Полное прохождение методички А. А. Друки: восемь этапов, включая **все семь блоков «Допишите код»**. В этой методичке нет отдельного индивидуального задания: итогом является классификация Fashion MNIST и визуализация результатов. Выполняйте ячейки сверху вниз.

## 1. Загрузка данных

Fashion MNIST: 60 000 обучающих и 10 000 тестовых изображений 28×28 пикселей, 10 классов одежды. Датасет загружается автоматически средствами Keras.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
tf.keras.utils.set_random_seed(42)
(x_train,y_train),(x_test,y_test)=fashion_mnist.load_data()
print("Train:",x_train.shape,y_train.shape)
print("Test:",x_test.shape,y_test.shape)
print("Пиксели:",x_train.min(),x_train.max())

## 2. Предобработка данных

### Допишите код №1: нормализация

Преобразуем целочисленные пиксели в float32 и делим на 255, чтобы получить значения от 0 до 1.

In [ ]:
x_train=x_train.astype("float32")/255.0
x_test=x_test.astype("float32")/255.0
print("После нормализации:",x_train.min(),x_train.max())

### Допишите код №2: one-hot кодирование меток

Для 10 классов получаем вектор длины 10. Оригинальные метки сохраним для подписей изображений.

In [ ]:
y_train_cat=to_categorical(y_train,num_classes=10)
y_test_cat=to_categorical(y_test,num_classes=10)
print("Метка:",y_train[0],"one-hot:",y_train_cat[0])

## 3. Вывод изображений на экран

Воспроизводим пример методички: первые 16 изображений, сетка 4×4 и подписи классов.

In [ ]:
plt.figure(figsize=(6,6))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.xticks([]);plt.yticks([])
    plt.imshow(x_train[i],cmap=plt.cm.binary)
    plt.xlabel(y_train[i])
plt.tight_layout();plt.show()

## 4. Реализация топологии нейронной сети

### Допишите код №3: Sequential, Flatten, Dense

Слой Flatten преобразует 28×28 в 784 признака, скрытые Dense с ReLU обучают нелинейные признаки, Softmax выдаёт вероятности десяти классов.

In [ ]:
model=Sequential([
    keras.Input(shape=(28,28)),
    Flatten(),
    Dense(128,activation="relu"),
    Dense(64,activation="relu"),
    Dense(10,activation="softmax")
])
model.summary()

## 5. Подготовка модели к обучению

### Допишите код №4: Adam и compile

Методичка обсуждает MSE; для one-hot меток и Softmax используем categorical crossentropy — подходящую функцию потерь для многоклассовой классификации.

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001),loss="categorical_crossentropy",metrics=["accuracy"])

## 6. Обучение

### Допишите код №5: fit

Обучаем сеть и сохраняем историю ошибки и точности. validation_split отделяет 20% обучающей выборки для проверки.

In [ ]:
history=model.fit(x_train,y_train_cat,epochs=10,batch_size=128,validation_split=0.2,verbose=1)

## 7. Тестирование

### Допишите код №6: evaluate

Тестовая выборка не участвует в обучении.

In [ ]:
loss,accuracy=model.evaluate(x_test,y_test_cat,verbose=0)
print(f"Test Loss: {loss:.4f}; Test Accuracy: {accuracy:.4f}")

## 8. Результат работы

### Допишите код №7: предсказания и классы

Получаем вероятности, индексы максимума и исходные метки.

In [ ]:
y_pred=model.predict(x_test,verbose=0)
y_pred_classes=np.argmax(y_pred,axis=1)
y_true=np.argmax(y_test_cat,axis=1)
print("Первые предсказания:",y_pred_classes[:10])
print("Правильные классы:",y_true[:10])

### 8.1. Пример методички: 15 случайных тестовых изображений

Зелёная подпись — правильный ответ; красная — ошибка.

In [ ]:
rng=np.random.default_rng(42)
plt.figure(figsize=(12,7))
random_indices=rng.choice(len(x_test),size=15,replace=False)
for i,idx in enumerate(random_indices):
    plt.subplot(3,5,i+1)
    plt.xticks([]);plt.yticks([]);plt.grid(False)
    plt.imshow(x_test[idx].reshape(28,28),cmap=plt.cm.binary)
    predicted_label=y_pred_classes[idx]
    true_label=y_true[idx]
    color="green" if predicted_label==true_label else "red"
    plt.title(f"Pred: {predicted_label}\nTrue: {true_label}",color=color,fontsize=11)
plt.tight_layout();plt.show()

### 8.2. Дополнительный анализ: история обучения и матрица ошибок

Эти графики помогают интерпретировать качество и обнаружить переобучение.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
for ax,key in zip(axs,["loss","accuracy"]):
    ax.plot(history.history[key],label="train")
    ax.plot(history.history["val_"+key],label="validation")
    ax.set(xlabel="Epoch",ylabel=key,title=key)
    ax.legend()
plt.tight_layout();plt.show()
print(classification_report(y_true,y_pred_classes))
ConfusionMatrixDisplay.from_predictions(y_true,y_pred_classes,cmap="Blues")
plt.show()

## 9. Вывод

После запуска укажите фактическую тестовую точность, классы, которые модель чаще путает, и разницу между обучающей и валидационной точностью. Все этапы методички выполнены; результаты обучения не подставлены заранее.